# Finviz Burst Precursors — Continuation Setup Research

Goal: find the optimal entry timing for a continuation setup on tickers that already appeared in Finviz Top Gainers.  
Entry is NOT at first TG appearance but AFTER a consolidation period (flag / volume dry-up / VWAP hold).

Walk-forward discipline: all feature selection uses TRAIN data only. Valid set untouched until Cell 13.

In [1]:
# Cell 1 — Imports & Config
import sqlite3
import pandas as pd
import numpy as np
from dateutil.parser import parse as dtparse
import pytz
import warnings
from scipy import stats

warnings.filterwarnings('ignore')

ET = pytz.timezone('America/New_York')

SNAP_DB  = '/Users/carlos/Library/Application Support/finviz-dashboard/finviz_snapshots.db'
BARS_DB  = '/Users/carlos/proyectos/TRADING/1_PROYECTOS_ACTIVOS/CLAUDE/trading_system_v3/trading_data.db'

TRAIN_END   = '2026-04-06'
CRASH_END   = '2026-04-10'
VALID_START = '2026-04-11'

print('Config loaded.')
print(f'  SNAP_DB : {SNAP_DB}')
print(f'  BARS_DB : {BARS_DB}')
print(f'  Train   : 2026-03-24 -> {TRAIN_END}')
print(f'  Crash   : 2026-04-07 -> {CRASH_END}  (excluded from main stats)')
print(f'  Valid   : {VALID_START} -> 2026-05-01')

Config loaded.
  SNAP_DB : /Users/carlos/Library/Application Support/finviz-dashboard/finviz_snapshots.db
  BARS_DB : /Users/carlos/proyectos/TRADING/1_PROYECTOS_ACTIVOS/CLAUDE/trading_system_v3/trading_data.db
  Train   : 2026-03-24 -> 2026-04-06
  Crash   : 2026-04-07 -> 2026-04-10  (excluded from main stats)
  Valid   : 2026-04-11 -> 2026-05-01


In [2]:
# Cell 2 — Load Finviz Top Gainers

def parse_volume_str(v):
    """Parse volume strings like '1.2M', '500K', '2B', '12345'."""
    if pd.isna(v) or v == '':
        return np.nan
    v = str(v).strip().replace(',', '')
    multipliers = {'B': 1e9, 'M': 1e6, 'K': 1e3}
    for suffix, mult in multipliers.items():
        if v.upper().endswith(suffix):
            try:
                return float(v[:-1]) * mult
            except ValueError:
                return np.nan
    try:
        return float(v)
    except ValueError:
        return np.nan

def parse_pct_str(p):
    """Parse percent strings like '+12.5%' or '-3.2%'."""
    if pd.isna(p) or p == '':
        return np.nan
    try:
        return float(str(p).replace('%', '').replace('+', '').strip())
    except ValueError:
        return np.nan

def parse_ts_et(ts_str):
    """Parse a timestamp string into an ET-aware datetime."""
    try:
        dt = dtparse(ts_str)
        if dt.tzinfo is None:
            dt = ET.localize(dt)
        else:
            dt = dt.astimezone(ET)
        return dt
    except Exception:
        return None

# Load snapshots  (category value is 'Top Gainers' — capital T, capital G)
con = sqlite3.connect(SNAP_DB)
snaps = pd.read_sql_query(
    "SELECT timestamp, category, ticker, price, change_pct, volume "
    "FROM snapshots WHERE category = 'Top Gainers'",
    con
)
con.close()
print(f'Raw snapshots loaded: {len(snaps):,} rows')

# Parse timestamps
snaps['ts'] = snaps['timestamp'].apply(parse_ts_et)
snaps = snaps[snaps['ts'].notna()].copy()

snaps['date']        = snaps['ts'].apply(lambda t: t.strftime('%Y-%m-%d'))
snaps['session_min'] = snaps['ts'].apply(
    lambda t: int((t.hour * 60 + t.minute) - (9 * 60 + 30))
)

# Parse numeric fields
snaps['price_f'] = pd.to_numeric(snaps['price'], errors='coerce')
snaps['chg_f']   = snaps['change_pct'].apply(parse_pct_str)
snaps['vol_f']   = snaps['volume'].apply(parse_volume_str)

# Filter to trading session
snaps = snaps[(snaps['session_min'] >= 0) & (snaps['session_min'] <= 390)].copy()

# Group by (ticker, date) -> first appearance
snaps_sorted = snaps.sort_values('ts')
first_app = snaps_sorted.groupby(['ticker', 'date']).first().reset_index()
n_apps    = snaps_sorted.groupby(['ticker', 'date']).size().reset_index(name='n_appearances')

tg_events = first_app.merge(n_apps, on=['ticker', 'date'])
tg_events = tg_events.rename(columns={
    'ts': 'first_ts', 'price_f': 'first_price',
    'chg_f': 'first_chg', 'session_min': 'first_session_min'
})

# Filters
tg_events = tg_events[
    (tg_events['first_session_min'] >= 10) &
    (tg_events['first_price'] >= 0.5) &
    (tg_events['first_price'] <= 30)
].copy()

# Keep only from analysis start
tg_events = tg_events[tg_events['date'] >= '2026-03-24'].copy()

print(f'tg_events after filters: {len(tg_events):,} events')
print(f'Date range: {tg_events["date"].min()} -> {tg_events["date"].max()}')
print(f'Unique tickers: {tg_events["ticker"].nunique()}')
print(tg_events[['ticker', 'date', 'first_price', 'first_chg', 'n_appearances']].head(5))

Raw snapshots loaded: 8,854 rows


tg_events after filters: 329 events
Date range: 2026-03-24 -> 2026-05-01
Unique tickers: 240
  ticker        date  first_price  first_chg  n_appearances
0   ABTS  2026-04-30         1.46      31.54              3
1   ACHV  2026-04-16         5.00      40.85              4
2   ADVB  2026-04-07         7.49      69.54              7
8   AGAE  2026-04-15         0.61      86.71             74
9   AGPU  2026-04-01         3.43     111.73              9


In [3]:
# Cell 3 — Load 1-min bars

def parse_bar_ts(ts_str):
    """Robustly parse bar timestamps (ISO T-format or space-separated) to ET."""
    if pd.isna(ts_str):
        return None
    try:
        dt = dtparse(str(ts_str))
        if dt.tzinfo is None:
            dt = ET.localize(dt)
        else:
            dt = dt.astimezone(ET)
        return dt
    except Exception:
        return None

print('Loading bars from DB (this may take a moment)...')
con = sqlite3.connect(BARS_DB)
bars_raw = pd.read_sql_query(
    """
    SELECT symbol, bar_timestamp, open_price, high_price, low_price, close_price, volume
    FROM market_intraday_bars
    WHERE bar_timestamp >= '2026-03-24'
    """,
    con
)
con.close()
print(f'Raw bars loaded: {len(bars_raw):,} rows, {bars_raw["symbol"].nunique()} symbols')

# Parse timestamps
print('Parsing timestamps...')
bars_raw['ts'] = bars_raw['bar_timestamp'].apply(parse_bar_ts)
bars_raw = bars_raw[bars_raw['ts'].notna()].copy()

# Filter to regular session: 9:30 AM - 4:00 PM ET
bars_raw['hour']   = bars_raw['ts'].apply(lambda t: t.hour)
bars_raw['minute'] = bars_raw['ts'].apply(lambda t: t.minute)
bars_raw['date']   = bars_raw['ts'].apply(lambda t: t.strftime('%Y-%m-%d'))

mask = (
    (bars_raw['hour'] > 9) |
    ((bars_raw['hour'] == 9) & (bars_raw['minute'] >= 30))
) & (
    (bars_raw['hour'] < 16)
)
bars_raw = bars_raw[mask].copy()

# Cast numeric columns
for col in ['open_price', 'high_price', 'low_price', 'close_price', 'volume']:
    bars_raw[col] = pd.to_numeric(bars_raw[col], errors='coerce')

bars_raw = bars_raw.dropna(subset=['open_price', 'high_price', 'low_price', 'close_price']).copy()
bars_raw = bars_raw.sort_values(['symbol', 'ts']).reset_index(drop=True)

# Build index: (symbol, date) -> DataFrame
print('Building bars index...')
bars_idx = {}
for (sym, dt), grp in bars_raw.groupby(['symbol', 'date']):
    bars_idx[(sym, dt)] = grp.reset_index(drop=True)

unique_pairs = len(bars_idx)
print(f'Bars index built: {unique_pairs:,} unique (symbol, date) pairs')
print(f'Date range: {bars_raw["date"].min()} -> {bars_raw["date"].max()}')

Loading bars from DB (this may take a moment)...


Raw bars loaded: 1,146,826 rows, 1049 symbols
Parsing timestamps...


Building bars index...


Bars index built: 2,877 unique (symbol, date) pairs


Date range: 2026-03-24 -> 2026-05-01


In [4]:
# Cell 4 — Build Universe

universe = []
skipped_no_bars = 0
skipped_ts_range = 0

for _, row in tg_events.iterrows():
    ticker   = row['ticker']
    date     = row['date']
    first_ts = row['first_ts']

    key = (ticker, date)
    if key not in bars_idx:
        skipped_no_bars += 1
        continue

    bdf = bars_idx[key]
    bar_min_ts = bdf['ts'].min()
    bar_max_ts = bdf['ts'].max()
    if first_ts < bar_min_ts or first_ts > bar_max_ts:
        skipped_ts_range += 1
        continue

    universe.append({
        'ticker':           ticker,
        'date':             date,
        'first_ts':         first_ts,
        'first_price':      row['first_price'],
        'first_chg':        row['first_chg'],
        'n_appearances':    row['n_appearances'],
        'first_session_min': row['first_session_min'],
    })

universe_df = pd.DataFrame(universe)
universe_df['period'] = 'crash'
universe_df.loc[universe_df['date'] <= TRAIN_END,   'period'] = 'train'
universe_df.loc[universe_df['date'] >= VALID_START, 'period'] = 'valid'

print(f'Universe: {len(universe_df):,} events')
print(f'  Skipped (no bars)  : {skipped_no_bars}')
print(f'  Skipped (ts range) : {skipped_ts_range}')
print()
print('By period:')
print(universe_df['period'].value_counts().sort_index())
print()
print('By date (first 10):')
print(universe_df.groupby('date').size().head(10))

Universe: 186 events
  Skipped (no bars)  : 143
  Skipped (ts range) : 0

By period:
period
crash    26
train    77
valid    83
Name: count, dtype: int64

By date (first 10):
date
2026-03-24     9
2026-03-25    17
2026-03-26    10
2026-03-27     8
2026-03-30     8
2026-04-01     7
2026-04-02     8
2026-04-06    10
2026-04-07     5
2026-04-08     6
dtype: int64


In [5]:
# Cell 5 — Compute VWAP for all bars

def compute_vwap(bars_df):
    """Compute cumulative VWAP from session open."""
    df = bars_df.copy()
    cum_pv  = (df['close_price'] * df['volume']).cumsum()
    cum_vol = df['volume'].cumsum()
    df['vwap'] = cum_pv / cum_vol.replace(0, np.nan)
    return df

print('Computing VWAP for all (symbol, date) pairs...')
bars_idx_vwap = {}
for key, bdf in bars_idx.items():
    bars_idx_vwap[key] = compute_vwap(bdf)

print(f'VWAP computed for {len(bars_idx_vwap):,} (symbol, date) pairs.')

# Quick sanity check
sample_key = list(bars_idx_vwap.keys())[0]
sample_df  = bars_idx_vwap[sample_key]
print(f'Sample key: {sample_key}')
print(sample_df[['ts', 'close_price', 'volume', 'vwap']].head(5))

Computing VWAP for all (symbol, date) pairs...


VWAP computed for 2,877 (symbol, date) pairs.
Sample key: ('AAL', '2026-04-17')
                         ts  close_price   volume       vwap
0 2026-04-17 09:30:00-04:00        12.91  2815011  12.910000
1 2026-04-17 09:31:00-04:00        13.02  1022028  12.939299
2 2026-04-17 09:32:00-04:00        12.98   759103  12.946022
3 2026-04-17 09:33:00-04:00        12.96   498728  12.947390
4 2026-04-17 09:34:00-04:00        13.03   801559  12.958620


In [6]:
# Cell 6 — Feature Computation

def compute_features(ticker, date, first_ts, bars_df):
    """
    Compute consolidation features from 20-bar window after first TG appearance.
    Returns dict or None if insufficient data.
    """
    after = bars_df[bars_df['ts'] > first_ts].copy()
    if len(after) < 5:
        return None

    # burst bar = first bar after signal
    burst = after.iloc[0]
    burst_range = float(burst['high_price'] - burst['low_price'])
    burst_vol   = float(burst['volume']) if burst['volume'] > 0 else np.nan

    # consolidation window = next 20 bars after burst
    consol = after.iloc[1:21] if len(after) > 20 else after.iloc[1:]
    if len(consol) < 3:
        return None

    consol_high    = float(consol['high_price'].max())
    consol_low     = float(consol['low_price'].min())
    consol_range   = consol_high - consol_low
    consol_avg_vol = float(consol['volume'].mean())

    burst_high   = float(burst['high_price'])
    pullback_pct = (burst_high - consol_low) / burst_high if burst_high > 0 else np.nan

    range_compression = consol_range / burst_range if burst_range > 0 else np.nan

    vol_dryup = consol_avg_vol / burst_vol if (burst_vol and burst_vol > 0) else np.nan

    if 'vwap' in consol.columns:
        valid_vwap = consol['vwap'].notna()
        if valid_vwap.sum() > 0:
            vwap_hold = float((consol.loc[valid_vwap, 'close_price'] >
                               consol.loc[valid_vwap, 'vwap']).mean())
        else:
            vwap_hold = np.nan
    else:
        vwap_hold = np.nan

    lows = consol['low_price'].values
    hl_formed = float(lows[-3:].min() > lows[:3].max()) if len(lows) >= 6 else np.nan

    last_bar = consol.iloc[-1]
    if 'vwap' in consol.columns and pd.notna(last_bar['vwap']) and last_bar['vwap'] > 0:
        price_over_vwap = float(last_bar['close_price'] / last_bar['vwap'] - 1)
    else:
        price_over_vwap = np.nan

    signal_session_min = int(
        (first_ts.hour * 60 + first_ts.minute) - (9 * 60 + 30)
    )

    consol_end_idx = consol.index[-1]

    return {
        'burst_range':        burst_range,
        'burst_vol':          burst_vol,
        'consol_high':        consol_high,
        'consol_low':         consol_low,
        'pullback_pct':       pullback_pct,
        'range_compression':  range_compression,
        'vol_dryup':          vol_dryup,
        'vwap_hold':          vwap_hold,
        'hl_formed':          hl_formed,
        'price_over_vwap':    price_over_vwap,
        'signal_session_min': signal_session_min,
        'consol_end_idx':     consol_end_idx,
    }

# Compute features for all universe events
feature_rows = []
skipped_feat = 0

for _, row in universe_df.iterrows():
    ticker   = row['ticker']
    date     = row['date']
    first_ts = row['first_ts']
    key      = (ticker, date)

    if key not in bars_idx_vwap:
        skipped_feat += 1
        continue

    bdf  = bars_idx_vwap[key]
    feat = compute_features(ticker, date, first_ts, bdf)
    if feat is None:
        skipped_feat += 1
        continue

    feat.update({
        'ticker':        ticker,
        'date':          date,
        'first_ts':      first_ts,
        'first_price':   row['first_price'],
        'first_chg':     row['first_chg'],
        'n_appearances': row['n_appearances'],
        'period':        row['period'],
    })
    feature_rows.append(feat)

features_df = pd.DataFrame(feature_rows)
print(f'Features computed: {len(features_df):,} events  |  skipped: {skipped_feat}')
print(f'\nFeature columns: {[c for c in features_df.columns if c not in ["ticker","date","first_ts","period"]]}')
print()
print('Non-null counts per feature:')
feat_cols = ['pullback_pct','range_compression','vol_dryup','vwap_hold','hl_formed','price_over_vwap']
print(features_df[feat_cols].notna().sum())

Features computed: 186 events  |  skipped: 0

Feature columns: ['burst_range', 'burst_vol', 'consol_high', 'consol_low', 'pullback_pct', 'range_compression', 'vol_dryup', 'vwap_hold', 'hl_formed', 'price_over_vwap', 'signal_session_min', 'consol_end_idx', 'first_price', 'first_chg', 'n_appearances']

Non-null counts per feature:
pullback_pct         186
range_compression    165
vol_dryup            171
vwap_hold            186
hl_formed            186
price_over_vwap      186
dtype: int64


In [7]:
# Cell 7 — Trade Simulation

def simulate_trade(
    entry_ts, ticker, date, bars_df,
    capital=200, stop_pct=0.06, trail_act=0.05, trail_dist=0.03
):
    """
    Simulate a trade:
    - Entry: open of first bar after entry_ts
    - Hard stop: 6% below entry
    - Trailing stop: activates at +5%; trails 3% below highest close
    - Exit: stop hit OR 15:58 ET EOD
    Returns dict or None.
    """
    after = bars_df[bars_df['ts'] > entry_ts].copy()
    if len(after) == 0:
        return None

    entry_bar = after.iloc[0]
    ep = float(entry_bar['open_price'])
    if ep <= 0:
        return None

    shares        = max(1, int(capital / ep))
    hard_stop     = ep * (1 - stop_pct)
    trail_active  = False
    trail_stop    = hard_stop
    highest_close = ep

    eod_cutoff = entry_bar['ts'].replace(hour=15, minute=58, second=0, microsecond=0)

    exit_price  = None
    exit_reason = None
    mfe = 0.0
    mae = 0.0

    for _, bar in after.iterrows():
        if bar['ts'] > eod_cutoff:
            exit_price  = float(bar['open_price'])
            exit_reason = 'EOD'
            break

        lo = float(bar['low_price'])
        hi = float(bar['high_price'])
        cl = float(bar['close_price'])

        mfe = max(mfe, (hi - ep) / ep)
        mae = min(mae, (lo - ep) / ep)

        if not trail_active and cl >= ep * (1 + trail_act):
            trail_active = True

        if trail_active:
            highest_close  = max(highest_close, cl)
            trail_stop     = highest_close * (1 - trail_dist)
            effective_stop = max(hard_stop, trail_stop)
        else:
            effective_stop = hard_stop

        if lo <= effective_stop:
            exit_price  = effective_stop
            exit_reason = 'TRAIL_STOP' if trail_active else 'HARD_STOP'
            break

    if exit_price is None:
        exit_price  = float(after.iloc[-1]['close_price'])
        exit_reason = 'EOD'

    ret = (exit_price - ep) / ep
    pnl = (exit_price - ep) * shares

    return {
        'ret':          ret,
        'pnl':          pnl,
        'mfe':          mfe,
        'mae':          mae,
        'exit_reason':  exit_reason,
        'ep':           ep,
        'shares':       shares,
        'trail_active': trail_active,
    }

def summarize_trades(trades_list, label=''):
    """Print WR, PF, avg_ret for a list of trade dicts."""
    if not trades_list:
        print(f'{label}: No trades.')
        return pd.DataFrame()
    df = pd.DataFrame(trades_list)
    n      = len(df)
    wr     = (df['ret'] > 0).mean()
    avg    = df['ret'].mean()
    wins   = df.loc[df['ret'] > 0, 'ret'].sum()
    losses = abs(df.loc[df['ret'] < 0, 'ret'].sum())
    pf     = wins / losses if losses > 0 else np.inf
    total_pnl = df['pnl'].sum()
    t_stat, p_val = stats.ttest_1samp(df['ret'], 0.0)
    sig = '***' if p_val < 0.001 else ('**' if p_val < 0.01 else ('*' if p_val < 0.05 else 'ns'))
    print(f'{label}: N={n:4d} | WR={wr:.1%} | AvgRet={avg:+.2%} | PF={pf:.2f} | PnL=${total_pnl:+.0f} | p={p_val:.3f} {sig}')
    return df

print('simulate_trade and summarize_trades defined.')

simulate_trade and summarize_trades defined.


In [8]:
# Cell 8 — Entry Strategy A: Immediate (Baseline)
# Enter at the very first bar after TG appearance

trades_A_all   = []
trades_A_train = []
trades_A_valid = []

for _, row in universe_df.iterrows():
    ticker   = row['ticker']
    date     = row['date']
    first_ts = row['first_ts']
    period   = row['period']
    key      = (ticker, date)

    if key not in bars_idx_vwap:
        continue

    bdf    = bars_idx_vwap[key]
    result = simulate_trade(first_ts, ticker, date, bdf)
    if result is None:
        continue

    result.update({'ticker': ticker, 'date': date, 'period': period})
    trades_A_all.append(result)
    if period == 'train':
        trades_A_train.append(result)
    elif period == 'valid':
        trades_A_valid.append(result)

print('=== Strategy A: Immediate Entry (Baseline) ===')
df_A_train = summarize_trades(trades_A_train, 'Train')
df_A_valid = summarize_trades(trades_A_valid, 'Valid')

if trades_A_all:
    df_A_all = pd.DataFrame(trades_A_all)
    print('\nExit breakdown (all periods):')
    print(df_A_all['exit_reason'].value_counts())
    print('\nTrain exit breakdown:')
    if len(df_A_train) > 0:
        print(df_A_train['exit_reason'].value_counts())

=== Strategy A: Immediate Entry (Baseline) ===
Train: N=  77 | WR=50.6% | AvgRet=-0.63% | PF=0.78 | PnL=$-101 | p=0.331 ns
Valid: N=  83 | WR=59.0% | AvgRet=+0.33% | PF=1.13 | PnL=$+52 | p=0.617 ns

Exit breakdown (all periods):
exit_reason
TRAIL_STOP    99
HARD_STOP     80
EOD            7
Name: count, dtype: int64

Train exit breakdown:
exit_reason
HARD_STOP     37
TRAIL_STOP    36
EOD            4
Name: count, dtype: int64


In [9]:
# Cell 9 — Entry Strategy B: Post-Consolidation Flag Breakout
# After 20-bar consolidation, enter when a bar closes above consol_high
# If no breakout within 40 bars, skip event

def detect_flag_breakout(ticker, date, first_ts, bars_df, consol_bars=20, lookforward=40):
    """
    Returns (entry_ts, entry_delay_min) or (None, None).
    consol_high is max high of bars [1..20] after first_ts.
    Breakout: bar closes above consol_high within next 40 bars.
    """
    after = bars_df[bars_df['ts'] > first_ts].copy()
    if len(after) < consol_bars + 2:
        return None, None

    consol      = after.iloc[1: consol_bars + 1]
    consol_high = float(consol['high_price'].max())

    scan_window = after.iloc[consol_bars + 1: consol_bars + 1 + lookforward]
    for _, bar in scan_window.iterrows():
        if float(bar['close_price']) > consol_high:
            delay_min = int((bar['ts'] - first_ts).total_seconds() / 60)
            return bar['ts'], delay_min

    return None, None

trades_B_all   = []
trades_B_train = []
trades_B_valid = []
delays_B       = []
no_breakout    = 0

for _, row in universe_df.iterrows():
    ticker   = row['ticker']
    date     = row['date']
    first_ts = row['first_ts']
    period   = row['period']
    key      = (ticker, date)

    if key not in bars_idx_vwap:
        continue

    bdf = bars_idx_vwap[key]
    bo_ts, delay = detect_flag_breakout(ticker, date, first_ts, bdf)

    if bo_ts is None:
        no_breakout += 1
        continue

    result = simulate_trade(bo_ts, ticker, date, bdf)
    if result is None:
        continue

    result.update({'ticker': ticker, 'date': date, 'period': period, 'delay_min': delay})
    trades_B_all.append(result)
    if period == 'train':
        trades_B_train.append(result)
    elif period == 'valid':
        trades_B_valid.append(result)
    delays_B.append(delay)

print('=== Strategy B: Flag Breakout Entry ===')
print(f'No breakout (skipped): {no_breakout}')
if delays_B:
    print(f'Avg entry delay: {np.mean(delays_B):.1f} min  |  Median: {np.median(delays_B):.1f} min')
print()
df_B_train = summarize_trades(trades_B_train, 'Train')
df_B_valid = summarize_trades(trades_B_valid, 'Valid')

=== Strategy B: Flag Breakout Entry ===
No breakout (skipped): 107
Avg entry delay: 33.9 min  |  Median: 32.0 min

Train: N=  31 | WR=67.7% | AvgRet=+1.14% | PF=1.68 | PnL=$+70 | p=0.208 ns
Valid: N=  36 | WR=44.4% | AvgRet=-1.32% | PF=0.55 | PnL=$-93 | p=0.110 ns


In [10]:
# Cell 10 — Entry Strategy C: Delayed Entry at T+20 (Simple Benchmark)
# Enter at bar 21 after signal — skip first 20 bars, no pattern detection

trades_C_all   = []
trades_C_train = []
trades_C_valid = []

for _, row in universe_df.iterrows():
    ticker   = row['ticker']
    date     = row['date']
    first_ts = row['first_ts']
    period   = row['period']
    key      = (ticker, date)

    if key not in bars_idx_vwap:
        continue

    bdf   = bars_idx_vwap[key]
    after = bdf[bdf['ts'] > first_ts].copy()

    if len(after) < 22:
        continue

    # entry_ts is the close time of bar 20, so next bar open = bar 21
    bar20_ts = after.iloc[20]['ts']
    result   = simulate_trade(bar20_ts, ticker, date, bdf)
    if result is None:
        continue

    result.update({'ticker': ticker, 'date': date, 'period': period})
    trades_C_all.append(result)
    if period == 'train':
        trades_C_train.append(result)
    elif period == 'valid':
        trades_C_valid.append(result)

print('=== Strategy C: T+20 Delayed Entry (Benchmark) ===')
df_C_train = summarize_trades(trades_C_train, 'Train')
df_C_valid = summarize_trades(trades_C_valid, 'Valid')

=== Strategy C: T+20 Delayed Entry (Benchmark) ===
Train: N=  76 | WR=52.6% | AvgRet=-0.31% | PF=0.88 | PnL=$-47 | p=0.611 ns
Valid: N=  82 | WR=52.4% | AvgRet=-0.50% | PF=0.81 | PnL=$-82 | p=0.433 ns


In [11]:
# Cell 11 — Feature Analysis (TRAIN SET ONLY)
# Bonferroni correction: 5 features -> significance threshold p < 0.01

BONFERRONI_THRESHOLD = 0.01  # 0.05 / 5 features

feat_train = features_df[features_df['period'] == 'train'].copy()
print(f'Train feature events: {len(feat_train)}')

# Run baseline trades for train feature events
feat_trade_rows = []
for _, row in feat_train.iterrows():
    ticker   = row['ticker']
    date     = row['date']
    first_ts = row['first_ts']
    key      = (ticker, date)
    if key not in bars_idx_vwap:
        continue
    bdf    = bars_idx_vwap[key]
    result = simulate_trade(first_ts, ticker, date, bdf)
    if result is None:
        continue
    r = dict(row)
    r['ret'] = result['ret']
    r['pnl'] = result['pnl']
    r['mfe'] = result['mfe']
    r['mae'] = result['mae']
    feat_trade_rows.append(r)

feat_trades_df = pd.DataFrame(feat_trade_rows)
print(f'Feature-trade events (train): {len(feat_trades_df)}')

feat_cols = ['pullback_pct', 'range_compression', 'vol_dryup', 'vwap_hold', 'hl_formed']
results_feat = []

print(f'\n=== Feature Analysis (Train only, Bonferroni p < {BONFERRONI_THRESHOLD}) ===')

for feat in feat_cols:
    col_data = feat_trades_df[[feat, 'ret']].dropna()
    if len(col_data) < 20:
        print(f'\n{feat}: insufficient data ({len(col_data)})')
        continue

    col_data = col_data.copy()
    # Use labels=False to get integer bin indices; then map to Q1-Q4 based on actual number of bins
    try:
        bin_codes, bin_edges = pd.qcut(col_data[feat], q=4, labels=False, retbins=True, duplicates='drop')
    except ValueError:
        print(f'\n{feat}: could not create quartiles (too many duplicate values), skipping.')
        continue

    n_bins = len(bin_edges) - 1
    if n_bins < 2:
        print(f'\n{feat}: only {n_bins} unique bin(s), skipping.')
        continue

    # Map integer bin codes to label strings based on actual bin count
    label_map = {i: f'Q{i+1}' for i in range(n_bins)}
    col_data['q'] = bin_codes.map(label_map)

    # Build the ordered list of labels actually present
    q_labels_present = [f'Q{i+1}' for i in range(n_bins)]
    # The "top" quartile label is the last one
    top_label = q_labels_present[-1]

    print(f'\n--- {feat} ({n_bins} bins) ---')
    print(f'{"Quartile":>10} | {"N":>5} | {"WR":>7} | {"AvgRet":>8} | {"PF":>6} | {"p-val":>8} | Sig')
    print('-' * 65)

    for q_label in q_labels_present:
        grp = col_data[col_data['q'] == q_label]['ret']
        n   = len(grp)
        if n < 3:
            print(f'{q_label:>10} | {n:>5} | -- too few --')
            continue
        wr     = (grp > 0).mean()
        avg    = grp.mean()
        wins   = grp[grp > 0].sum()
        losses = abs(grp[grp < 0].sum())
        pf     = wins / losses if losses > 0 else np.inf
        t_stat, p_val = stats.ttest_1samp(grp, 0.0)
        sig = '***' if p_val < 0.001 else ('**' if p_val < BONFERRONI_THRESHOLD else ('*' if p_val < 0.05 else 'ns'))
        print(f'{q_label:>10} | {n:>5} | {wr:>7.1%} | {avg:>+8.2%} | {pf:>6.2f} | {p_val:>8.4f} | {sig}')
        results_feat.append({'feature': feat, 'quartile': q_label, 'n': n, 'wr': wr, 'avg_ret': avg, 'pf': pf, 'p_val': p_val})

    top_q = col_data[col_data['q'] == top_label]['ret']
    rest  = col_data[col_data['q'] != top_label]['ret']
    if len(top_q) >= 3 and len(rest) >= 3:
        t2, p2 = stats.ttest_ind(top_q, rest)
        marker = '<<< SIGNAL' if p2 < BONFERRONI_THRESHOLD else ''
        print(f'  Top quartile ({top_label}) vs rest: t={t2:.2f}, p={p2:.4f} {marker}')

print(f'\nNote: Significance threshold after Bonferroni correction = p < {BONFERRONI_THRESHOLD} (0.05/5 features)')

Train feature events: 77


Feature-trade events (train): 77

=== Feature Analysis (Train only, Bonferroni p < 0.01) ===

--- pullback_pct (4 bins) ---
  Quartile |     N |      WR |   AvgRet |     PF |    p-val | Sig
-----------------------------------------------------------------


        Q1 |    20 |   90.0% |   +3.22% |   6.37 |   0.0027 | **
        Q2 |    19 |   63.2% |   +0.57% |   1.28 |   0.6297 | ns
        Q3 |    19 |   42.1% |   -1.01% |   0.71 |   0.5131 | ns
        Q4 |    19 |    5.3% |   -5.52% |   0.03 |   0.0000 | ***
  Top quartile (Q4) vs rest: t=-4.96, p=0.0000 <<< SIGNAL

--- range_compression (4 bins) ---
  Quartile |     N |      WR |   AvgRet |     PF |    p-val | Sig
-----------------------------------------------------------------
        Q1 |    18 |   38.9% |   -2.41% |   0.34 |   0.0464 | *
        Q2 |    17 |   47.1% |   -0.98% |   0.69 |   0.4808 | ns
        Q3 |    17 |   70.6% |   +1.02% |   1.66 |   0.3599 | ns
        Q4 |    18 |   55.6% |   +0.62% |   1.23 |   0.7095 | ns
  Top quartile (Q4) vs rest: t=0.94, p=0.3498 

--- vol_dryup (4 bins) ---
  Quartile |     N |      WR |   AvgRet |     PF |    p-val | Sig
-----------------------------------------------------------------
        Q1 |    18 |   33.3% |   -2.47% |   0.3

In [12]:
# Cell 12 — Best Filter Combination (TRAIN ONLY)

print('=== Best Filter Combination (selected on TRAIN data) ===')
print()

if results_feat:
    rf_df = pd.DataFrame(results_feat)

    feat_summary = []
    for feat in feat_cols:
        fdf = rf_df[rf_df['feature'] == feat]
        if fdf.empty:
            continue
        # Find the top quartile label (highest Q number present for this feature)
        quartiles_present = sorted(fdf['quartile'].unique(), key=lambda x: int(x[1:]))
        top_q_label = quartiles_present[-1]
        bot_q_label = quartiles_present[0]

        q4 = fdf[fdf['quartile'] == top_q_label]
        q1 = fdf[fdf['quartile'] == bot_q_label]
        if q4.empty:
            continue
        q4_avg  = float(q4['avg_ret'].values[0])
        q4_pf   = float(q4['pf'].values[0])
        q4_pval = float(q4['p_val'].values[0])
        q1_avg  = float(q1['avg_ret'].values[0]) if not q1.empty else np.nan
        spread  = q4_avg - (q1_avg if not np.isnan(q1_avg) else 0)
        feat_summary.append({
            'feature':    feat,
            'q4_avg_ret': q4_avg,
            'q4_pf':      q4_pf,
            'q4_pval':    q4_pval,
            'q1q4_spread': spread,
            'signal':     q4_pval < BONFERRONI_THRESHOLD,
        })

    feat_summary_df = pd.DataFrame(feat_summary).sort_values('q4_avg_ret', ascending=False)
    print('Feature ranking by top-quartile avg_ret (train):')
    print(feat_summary_df[['feature','q4_avg_ret','q4_pf','q4_pval','signal']].to_string(index=False))
    print()

    signal_feats = feat_summary_df[feat_summary_df['signal'] == True]['feature'].tolist()
    top_feats    = feat_summary_df.head(3)['feature'].tolist()

    if signal_feats:
        chosen_feats = signal_feats[:3]
        print(f'Features with Bonferroni-significant signal: {signal_feats}')
        print(f'Using: {chosen_feats}')
    else:
        chosen_feats = top_feats[:2]
        print(f'No Bonferroni-significant features. Using top-2 by avg_ret for exploratory analysis:')
        print(f'  {chosen_feats}')
        print('  WARNING: These filters are NOT statistically validated -- do NOT trade live.')

else:
    print('No feature results available. Using empty filter set.')
    chosen_feats = []

print()
print('This combination was chosen on TRAIN data and will be validated below.')

# Compute train medians for chosen features
train_medians = {}
for feat in chosen_feats:
    med = feat_trades_df[feat].median()
    train_medians[feat] = med
    print(f'  {feat}: train median = {med:.4f}')

def apply_filter(df, feat_list, medians):
    """Keep rows where all features are >= their train median."""
    mask = pd.Series([True] * len(df), index=df.index)
    for feat in feat_list:
        if feat in medians:
            mask = mask & (df[feat] >= medians[feat])
    return df[mask]

print()
if chosen_feats:
    filtered_train = apply_filter(feat_trades_df, chosen_feats, train_medians)
    print(f'Train events passing filter: {len(filtered_train)} / {len(feat_trades_df)}')
    if len(filtered_train) >= 5:
        _ = summarize_trades(filtered_train.to_dict('records'), f'Best Filter (Train, n={len(filtered_train)})')
    else:
        print('Too few events to summarize.')
else:
    print('No filter applied (no chosen features).')
    filtered_train = feat_trades_df

=== Best Filter Combination (selected on TRAIN data) ===

Feature ranking by top-quartile avg_ret (train):
          feature  q4_avg_ret    q4_pf      q4_pval  signal
        vol_dryup    0.023855 3.123036 7.929381e-02   False
        vwap_hold    0.017101 2.016847 7.619047e-02   False
range_compression    0.006220 1.233232 7.095422e-01   False
     pullback_pct   -0.055173 0.029370 1.101500e-09    True

Features with Bonferroni-significant signal: ['pullback_pct']
Using: ['pullback_pct']

This combination was chosen on TRAIN data and will be validated below.
  pullback_pct: train median = 0.0577

Train events passing filter: 39 / 77


Best Filter (Train, n=39): N=  39 | WR=23.1% | AvgRet=-3.33% | PF=0.28 | PnL=$-257 | p=0.000 ***


In [13]:
# Cell 13 — Walk-Forward Validation
# Apply EXACT filter from Cell 12 to VALID set (Apr 11 - May 1)

print('=== Walk-Forward Validation (VALID SET: Apr 11 - May 1) ===')
print(f'Filters applied: {chosen_feats}')
for feat, med in train_medians.items():
    print(f'  {feat} >= {med:.4f}  (computed on train data only)')
print()

feat_valid = features_df[features_df['period'] == 'valid'].copy()

# Run baseline trades for valid feature events
valid_trade_rows = []
for _, row in feat_valid.iterrows():
    ticker   = row['ticker']
    date     = row['date']
    first_ts = row['first_ts']
    key      = (ticker, date)
    if key not in bars_idx_vwap:
        continue
    bdf    = bars_idx_vwap[key]
    result = simulate_trade(first_ts, ticker, date, bdf)
    if result is None:
        continue
    r = dict(row)
    r['ret'] = result['ret']
    r['pnl'] = result['pnl']
    r['mfe'] = result['mfe']
    r['mae'] = result['mae']
    valid_trade_rows.append(r)

feat_valid_trades_df = pd.DataFrame(valid_trade_rows)
print(f'Valid feature-trade events (all): {len(feat_valid_trades_df)}')

def pf_score(trade_dicts_or_df):
    if isinstance(trade_dicts_or_df, list):
        df = pd.DataFrame(trade_dicts_or_df)
    else:
        df = trade_dicts_or_df
    if df.empty or 'ret' not in df.columns:
        return np.nan
    wins   = df.loc[df['ret'] > 0, 'ret'].sum()
    losses = abs(df.loc[df['ret'] < 0, 'ret'].sum())
    return wins / losses if losses > 0 else np.inf

if chosen_feats and train_medians:
    filtered_valid = apply_filter(feat_valid_trades_df, chosen_feats, train_medians)
    print(f'Valid events passing filter: {len(filtered_valid)}')
    print()

    train_pf = pf_score(filtered_train)

    if len(filtered_valid) >= 5:
        df_fv    = summarize_trades(filtered_valid.to_dict('records'), f'Best Filter (Valid, n={len(filtered_valid)})')
        valid_pf = pf_score(filtered_valid)

        print()
        print(f'Train PF: {train_pf:.2f}')
        print(f'Valid PF: {valid_pf:.2f}')

        if not np.isinf(train_pf) and valid_pf < 0.5 * train_pf:
            print('OVERFITTING WARNING: Valid PF < 0.5 * Train PF -- likely overfit to train period.')
        elif not np.isinf(valid_pf) and valid_pf >= 1.0:
            print('PROMISING: Valid PF >= 1.0 -- edge held out-of-sample.')
        else:
            print('INCONCLUSIVE: insufficient signal or small sample.')
    else:
        print(f'Too few valid events ({len(filtered_valid)}) to validate.')
        filtered_valid = pd.DataFrame()
else:
    print('No filter to validate (no significant features on train).')
    print()
    print('Valid baseline (unfiltered):')
    filtered_valid = feat_valid_trades_df
    if len(feat_valid_trades_df) >= 5:
        _ = summarize_trades(feat_valid_trades_df.to_dict('records'), f'Baseline (Valid, n={len(feat_valid_trades_df)})')

=== Walk-Forward Validation (VALID SET: Apr 11 - May 1) ===
Filters applied: ['pullback_pct']
  pullback_pct >= 0.0577  (computed on train data only)



Valid feature-trade events (all): 83
Valid events passing filter: 37

Best Filter (Valid, n=37): N=  37 | WR=24.3% | AvgRet=-3.33% | PF=0.27 | PnL=$-243 | p=0.000 ***

Train PF: 0.28
Valid PF: 0.27
INCONCLUSIVE: insufficient signal or small sample.


In [14]:
# Cell 14 — Summary Table

def row_stats(trade_dicts, label, period):
    """Return a summary dict for the table."""
    if not trade_dicts:
        return {'Strategy': label, 'Period': period, 'N': 0,
                'WR': '-', 'AvgRet': '-', 'PF': '-', 'Sig': '-'}
    df = pd.DataFrame(trade_dicts)
    n      = len(df)
    wr     = (df['ret'] > 0).mean()
    avg    = df['ret'].mean()
    wins   = df.loc[df['ret'] > 0, 'ret'].sum()
    losses = abs(df.loc[df['ret'] < 0, 'ret'].sum())
    pf     = wins / losses if losses > 0 else np.inf
    _, p   = stats.ttest_1samp(df['ret'], 0.0)
    sig    = '***' if p < 0.001 else ('**' if p < 0.01 else ('*' if p < 0.05 else 'ns'))
    return {
        'Strategy': label, 'Period': period, 'N': n,
        'WR':     f'{wr:.1%}',
        'AvgRet': f'{avg:+.2%}',
        'PF':     f'{pf:.2f}' if not np.isinf(pf) else 'inf',
        'Sig':    sig,
    }

# Collect filtered train/valid lists
filtered_train_list = filtered_train.to_dict('records') if isinstance(filtered_train, pd.DataFrame) and len(filtered_train) > 0 else []
filtered_valid_list = filtered_valid.to_dict('records') if isinstance(filtered_valid, pd.DataFrame) and len(filtered_valid) > 0 else []

filter_label = f'Best Filter ({" + ".join(chosen_feats) if chosen_feats else "none"})'

table_rows = [
    row_stats(trades_A_train, 'Baseline (immediate)', 'Train'),
    row_stats(trades_A_valid, 'Baseline (immediate)', 'Valid'),
    row_stats(trades_B_train, 'Flag Breakout',        'Train'),
    row_stats(trades_B_valid, 'Flag Breakout',        'Valid'),
    row_stats(trades_C_train, 'T+20 Delayed',         'Train'),
    row_stats(trades_C_valid, 'T+20 Delayed',         'Valid'),
    row_stats(filtered_train_list, filter_label,      'Train'),
    row_stats(filtered_valid_list, filter_label,      'Valid'),
]

summary_table = pd.DataFrame(table_rows)

print('=== Strategy Comparison Summary ===')
print()
print(summary_table.to_string(index=False))
print()
print('Significance: *** p<0.001, ** p<0.01, * p<0.05, ns = not significant')
print('Crash week (2026-04-07 to 2026-04-10) excluded from all metrics.')

=== Strategy Comparison Summary ===

                  Strategy Period  N    WR AvgRet   PF Sig
      Baseline (immediate)  Train 77 50.6% -0.63% 0.78  ns
      Baseline (immediate)  Valid 83 59.0% +0.33% 1.13  ns
             Flag Breakout  Train 31 67.7% +1.14% 1.68  ns
             Flag Breakout  Valid 36 44.4% -1.32% 0.55  ns
              T+20 Delayed  Train 76 52.6% -0.31% 0.88  ns
              T+20 Delayed  Valid 82 52.4% -0.50% 0.81  ns
Best Filter (pullback_pct)  Train 39 23.1% -3.33% 0.28 ***
Best Filter (pullback_pct)  Valid 37 24.3% -3.33% 0.27 ***

Significance: *** p<0.001, ** p<0.01, * p<0.05, ns = not significant
Crash week (2026-04-07 to 2026-04-10) excluded from all metrics.


In [15]:
# Cell 15 — Conclusions

print('=== CONCLUSIONS ===')
print()

def pf_val(trade_dicts):
    if not trade_dicts:
        return np.nan
    df = pd.DataFrame(trade_dicts)
    wins   = df.loc[df['ret'] > 0, 'ret'].sum()
    losses = abs(df.loc[df['ret'] < 0, 'ret'].sum())
    return wins / losses if losses > 0 else np.inf

def p_val_func(trade_dicts):
    if not trade_dicts:
        return 1.0
    df = pd.DataFrame(trade_dicts)
    _, p = stats.ttest_1samp(df['ret'], 0.0)
    return p

# 1. Is the edge real?
print('1. IS THE EDGE REAL?')
filt_tp = p_val_func(filtered_train_list)
filt_vp = p_val_func(filtered_valid_list)
filt_vpf = pf_val(filtered_valid_list)

if filt_tp < 0.01 and filt_vp < 0.05 and (np.isinf(filt_vpf) or filt_vpf >= 1.0):
    print('   YES -- filter shows p<0.01 on train and p<0.05 on valid with PF>=1.0.')
elif filt_tp < 0.01:
    print('   PARTIAL -- filter significant on train but not confirmed on valid.')
    print('   Possible overfitting or insufficient valid sample.')
else:
    print('   NO CONFIRMED EDGE -- neither baseline nor best filter reached p<0.01 on train.')
    print('   Do not trade live based on these results.')
print()

# 2. Best entry strategy
print('2. BEST ENTRY STRATEGY')
pf_a = pf_val(trades_A_train)
pf_b = pf_val(trades_B_train)
pf_c = pf_val(trades_C_train)
strategies = [('Baseline (immediate)', pf_a), ('Flag Breakout', pf_b), ('T+20 Delayed', pf_c)]
best_strat = max(strategies, key=lambda x: x[1] if not np.isnan(x[1]) else -1)
print(f'   Best by train PF: {best_strat[0]} (PF={best_strat[1]:.2f})')
print(f'   Baseline PF: {pf_a:.2f}  |  Flag Breakout PF: {pf_b:.2f}  |  T+20 PF: {pf_c:.2f}')
print()

# 3. Features that matter
print('3. FEATURES THAT MATTER (train analysis only)')
if results_feat:
    rf_df2 = pd.DataFrame(results_feat)
    sig_rows = rf_df2[rf_df2['p_val'] < BONFERRONI_THRESHOLD][['feature','quartile','n','avg_ret','pf','p_val']]
    if len(sig_rows) > 0:
        print('   Statistically significant (Bonferroni p<0.01) feature-quartile combinations:')
        print(sig_rows.to_string(index=False))
    else:
        print('   No feature-quartile combination reached Bonferroni significance (p<0.01).')
        print('   Top features by top-quartile avg_ret (exploratory only):')
        # Get the top-quartile row for each feature (highest Q label)
        top_q_rows = []
        for feat in feat_cols:
            fdf2 = rf_df2[rf_df2['feature'] == feat]
            if fdf2.empty:
                continue
            top_lbl = sorted(fdf2['quartile'].unique(), key=lambda x: int(x[1:]))[-1]
            top_q_rows.append(fdf2[fdf2['quartile'] == top_lbl])
        if top_q_rows:
            top_by_ret = pd.concat(top_q_rows).sort_values('avg_ret', ascending=False).head(3)
            print(top_by_ret[['feature','avg_ret','pf','p_val']].to_string(index=False))
else:
    print('   No feature analysis data available.')
print()

# 4. Minimum N to trade live
print('4. MINIMUM N TO TRADE LIVE')
print('   Rule of thumb: need 50+ trades with p<0.01 on train AND valid confirmation.')
n_filt_train = len(filtered_train_list)
n_filt_valid = len(filtered_valid_list)
print(f'   Current filter: {n_filt_train} train, {n_filt_valid} valid.')
if n_filt_train < 30:
    print('   NOT ready for live trading -- insufficient sample size on train.')
elif n_filt_valid < 15:
    print('   NOT ready -- insufficient validation sample. Collect more data before trading live.')
else:
    print('   Sample size may be adequate if edge is confirmed. Monitor slippage in live trading.')
print()

# 5. What to monitor
print('5. WHAT TO MONITOR GOING FORWARD')
print('   - Weekly PF rolling on valid period -- flag if it drops below 1.0 for 3 consecutive weeks')
print('   - Vol dry-up ratio: check if distribution shifts (regime change in liquidity)')
print('   - VWAP hold rate: correlated with broader market conditions')
print('   - Crash sensitivity: Liberation Day crash (Apr 7-10) caused drawdowns on TG tickers')
print('     -> Add SPY regime filter ONLY as kill-switch (not as signal filter)')
print('   - Expand valid period: re-run every 2 weeks as new data accumulates')
print()
print('Done.')

=== CONCLUSIONS ===

1. IS THE EDGE REAL?
   PARTIAL -- filter significant on train but not confirmed on valid.
   Possible overfitting or insufficient valid sample.

2. BEST ENTRY STRATEGY
   Best by train PF: Flag Breakout (PF=1.68)
   Baseline PF: 0.78  |  Flag Breakout PF: 1.68  |  T+20 PF: 0.88

3. FEATURES THAT MATTER (train analysis only)
   Statistically significant (Bonferroni p<0.01) feature-quartile combinations:
     feature quartile  n   avg_ret       pf        p_val
pullback_pct       Q1 20  0.032233 6.372141 2.736611e-03
pullback_pct       Q4 19 -0.055173 0.029370 1.101500e-09
   vwap_hold       Q1 40 -0.027974 0.309283 6.192356e-04

4. MINIMUM N TO TRADE LIVE
   Rule of thumb: need 50+ trades with p<0.01 on train AND valid confirmation.
   Current filter: 39 train, 37 valid.
   Sample size may be adequate if edge is confirmed. Monitor slippage in live trading.

5. WHAT TO MONITOR GOING FORWARD
   - Weekly PF rolling on valid period -- flag if it drops below 1.0 for 3 co